In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS anvisa
COMMENT 'Schema unificado para o projeto Anvisa (Hands-on Mackenzie)';

In [0]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import current_timestamp, col

# 1. Parâmetros de infraestrutura
storage_account = "stmarcelodatalab"
container = "projetos"
projeto_dir = "hands_on_mackenzie"
schema_nome = "anvisa"

# Garante a existência do schema no catálogo atual
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_nome}")

# 2. Localização dos arquivos brutos no Workspace
workspace_user_path = "file:/Workspace/Users/marcelofranco@outlook.com.br/hands-on-mackenzie-v2"

# 3. Relação de tabelas com prefixo b_
arquivos_para_ingerir = {
    "DADOS_ABERTOS_MEDICAMENTOS.csv": "b_dados_abertos_medicamentos",
    "VigiMed_Medicamentos.csv": "b_vigimed_medicamentos",
    "VigiMed_Notificacoes.csv": "b_vigimed_notificacoes",
    "VigiMed_Reacoes.csv": "b_vigimed_reacoes"
}

# 4. Pipeline de Ingestão
for arquivo_csv, tabela_nome in arquivos_para_ingerir.items():
    caminho_origem_csv = f"{workspace_user_path}/{arquivo_csv}"
    caminho_destino_bronze = (
        f"abfss://{container}@{storage_account}.dfs.core.windows.net/"
        f"{projeto_dir}/01_bronze/{tabela_nome}"
    )
    
    print(f"--- Processando: {arquivo_csv} -> {schema_nome}.{tabela_nome} ---")
    
    # Passo A: Extrai a lista de colunas a partir do cabeçalho
    df_temp = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("multiLine", "true")    # Permite quebras de linha dentro do mesmo campo
        .option("escape", '"')          
        .load(caminho_origem_csv)
    )
    
    # Passo B: Monta schema forçando 100% StringType
    schema_all_string = StructType([
        StructField(col_name, StringType(), True) 
        for col_name in df_temp.columns
    ])
    
    # Passo C: Leitura completa aplicando o schema explícito
    df_raw = (
        spark.read.format("csv")
        .option("header", "true")
        .option("sep", ";")
        .option("encoding", "ISO-8859-1")
        .schema(schema_all_string)
        .load(caminho_origem_csv)
    )
    
    # Passo D: Metadados técnicos suportados no Unity Catalog
    df_bronze = (
        df_raw
        .withColumn("_data_ingestao", current_timestamp())
        .withColumn("_arquivo_origem", col("_metadata.file_path"))
    )
    
    # Passo E: Grava fisicamente no ADLS Gen2 na pasta 01_bronze
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .save(caminho_destino_bronze)
    )
    
    # Passo F: Registra como tabela no catálogo Unity sem conflito de External Location
    (
        df_bronze.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{schema_nome}.{tabela_nome}")
    )
    
    print(f"Sucesso: Tabela '{schema_nome}.{tabela_nome}' registrada e arquivos salvos em '{caminho_destino_bronze}'\n")

--- Processando: DADOS_ABERTOS_MEDICAMENTOS.csv -> anvisa.b_dados_abertos_medicamentos ---
Sucesso: Tabela 'anvisa.b_dados_abertos_medicamentos' registrada e arquivos salvos em 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/01_bronze/b_dados_abertos_medicamentos'

--- Processando: VigiMed_Medicamentos.csv -> anvisa.b_vigimed_medicamentos ---
Sucesso: Tabela 'anvisa.b_vigimed_medicamentos' registrada e arquivos salvos em 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/01_bronze/b_vigimed_medicamentos'

--- Processando: VigiMed_Notificacoes.csv -> anvisa.b_vigimed_notificacoes ---
Sucesso: Tabela 'anvisa.b_vigimed_notificacoes' registrada e arquivos salvos em 'abfss://projetos@stmarcelodatalab.dfs.core.windows.net/hands_on_mackenzie/01_bronze/b_vigimed_notificacoes'

--- Processando: VigiMed_Reacoes.csv -> anvisa.b_vigimed_reacoes ---
Sucesso: Tabela 'anvisa.b_vigimed_reacoes' registrada e arquivos salvos em 'abfss://projetos@stmarce

In [0]:
import json
from pyspark.sql import functions as F

def gerar_perfil_tabela(nome_tabela):
    print(f"Iniciando o perfil da tabela: {nome_tabela}")
    df = spark.table(nome_tabela)

    # 1. Agregação eficiente dos valores distintos e nulos (dispara apenas 1 Job)
    exprs = []
    for c in df.columns:
        exprs.append(F.approx_count_distinct(F.col(c)).alias(f"{c}_distinct"))
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls"))

    agg_results = df.agg(*exprs).collect()[0].asDict()

    # 2. Montar dicionário estruturado
    perfil_tabela = {
        "tabela": nome_tabela,
        "dicionario_de_dados": [
            {
                "coluna": col,
                "tipo_de_dado": dtype,
                "valores_distintos_aprox": agg_results[f"{col}_distinct"],
                "quantidade_nulos": agg_results[f"{col}_nulls"]
            }
            for col, dtype in df.dtypes
        ],
        "amostra_dados": [row.asDict() for row in df.limit(5).collect()]
    }

    # 3. Formatar em JSON
    # O uso de default=str é vital para evitar erros com datas ou decimais do PySpark
    resultado_llm = json.dumps(perfil_tabela, indent=2, default=str, ensure_ascii=False)

    # 4. Salvar o resultado em um arquivo JSON dinâmico
    # Substituímos pontos por underlines para evitar problemas com o nome do arquivo
    nome_arquivo_seguro = nome_tabela.replace(".", "_")
    nome_arquivo = f"{nome_arquivo_seguro}_schema.json"
    
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(resultado_llm)

    print(f"Arquivo '{nome_arquivo}' salvo com sucesso!\n")

# Lista de tabelas que você deseja processar
tabelas = [
    "anvisa.b_dados_abertos_medicamentos",
    "anvisa.b_vigimed_medicamentos",
    "anvisa.b_vigimed_notificacoes",
    "anvisa.b_vigimed_reacoes"
]

# Executa a função para cada tabela na lista
for tb in tabelas:
    gerar_perfil_tabela(tb)

Iniciando o perfil da tabela: anvisa.b_dados_abertos_medicamentos
Arquivo 'anvisa_b_dados_abertos_medicamentos_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.b_vigimed_medicamentos
Arquivo 'anvisa_b_vigimed_medicamentos_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.b_vigimed_notificacoes
Arquivo 'anvisa_b_vigimed_notificacoes_schema.json' salvo com sucesso!

Iniciando o perfil da tabela: anvisa.b_vigimed_reacoes
Arquivo 'anvisa_b_vigimed_reacoes_schema.json' salvo com sucesso!



In [0]:
import json
from pyspark.sql import functions as F

def gerar_perfil_tabela(nome_tabela):
    print(f"Iniciando o perfil da tabela: {nome_tabela}")
    df = spark.table(nome_tabela)

    # 1. Agregação eficiente dos valores distintos e nulos (dispara apenas 1 Job)
    exprs = []
    for c in df.columns:
        exprs.append(F.approx_count_distinct(F.col(c)).alias(f"{c}_distinct"))
        exprs.append(F.sum(F.col(c).isNull().cast("int")).alias(f"{c}_nulls"))

    agg_results = df.agg(*exprs).collect()[0].asDict()

    # 2. Montar dicionário estruturado
    perfil_tabela = {
        "tabela": nome_tabela,
        "dicionario_de_dados": [
            {
                "coluna": col,
                "tipo_de_dado": dtype,
                "valores_distintos_aprox": agg_results[f"{col}_distinct"],
                "quantidade_nulos": agg_results[f"{col}_nulls"]
            }
            for col, dtype in df.dtypes
        ],
        "amostra_dados": [row.asDict() for row in df.limit(5).collect()]
    }

    # 3. Formatar em JSON
    # O uso de default=str é vital para evitar erros com datas ou decimais do PySpark
    resultado_llm = json.dumps(perfil_tabela, indent=2, default=str, ensure_ascii=False)

    # 4. Salvar o resultado em um arquivo JSON dinâmico
    # Substituímos pontos por underlines para evitar problemas com o nome do arquivo
    nome_arquivo_seguro = nome_tabela.replace(".", "_")
    nome_arquivo = f"{nome_arquivo_seguro}_schema.json"
    
    with open(nome_arquivo, "w", encoding="utf-8") as f:
        f.write(resultado_llm)

    print(f"Arquivo '{nome_arquivo}' salvo com sucesso!\n")

# Lista de tabelas que você deseja processar
tabelas = [
    "anvisa.b_dados_abertos_medicamentos"
]

# Executa a função para cada tabela na lista
for tb in tabelas:
    gerar_perfil_tabela(tb)

Iniciando o perfil da tabela: anvisa.b_dados_abertos_medicamentos
Arquivo 'anvisa_b_dados_abertos_medicamentos_schema.json' salvo com sucesso!

